# 01 - Data Collection

**Goal:** load the raw IEC LGE 2016 and 2021 results files (one CSV per province, per year) into two combined DataFrames, and do a first pass of sanity checks before any cleaning happens.

**Source data:**
- 2016 LGE results - https://results.elections.org.za/home/downloads/me-results
- 2021 LGE results - https://results.elections.org.za/home/downloads/me-results

**Expected raw layout:**
```
data/raw/LGE2016/<PROVINCE_CODE>.csv   e.g. EC.csv, FS.csv, GT.csv ...
data/raw/LGE2021/<PROVINCE_CODE>.csv
```
Each file is at **voting-station × party** grain - one row per party per voting station, with station-level fields (`RegisteredVoters`, `SpoiltVotes`, `BallotType`) repeated across every party row for that station. That repetition is handled in `02_cleaning_and_merge`, not here.

In [ ]:
import glob
import pandas as pd

pd.set_option("display.max_columns", None)

## Load all province files for each year

In [ ]:
def load_province_files(year_dir: str) -> pd.DataFrame:
    """Load and concatenate every per-province CSV in a year's raw data folder."""
    files = sorted(glob.glob(f"{year_dir}/*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {year_dir}")
    print(f"{year_dir}: found {len(files)} province files")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

In [ ]:
df_2016 = load_province_files("../data/raw/LGE2016")
df_2021 = load_province_files("../data/raw/LGE2021")

print("2016 rows:", len(df_2016))
print("2021 rows:", len(df_2021))

> Note: Raw row counts nearly doubled from 2016 to 2021 (653k → 1,084k rows). This was investigated and found to be explained by growth in contesting parties/candidates (206 → 324), not duplication - voting district counts stayed stable (22,612 → 23,147, ~2% growth).

### Structure

In [ ]:
df_2016.head()

In [ ]:
df_2016.info()

In [ ]:
df_2021.head()

In [ ]:
df_2021.info()

### Sanity checks

In [ ]:
# Ballot types present - LGE ballots include Ward, PR, and (in local
# municipalities only) DC 40%. We deliberately keep all of them here;
# 02_cleaning_and_merge filters to a single ballot type before aggregating.
print("2016 BallotType values:", df_2016["BallotType"].unique())
print("2021 BallotType values:", df_2021["BallotType"].unique())


In [ ]:
# Missing values check
print("2016 missing values:\n", df_2016.isna().sum())
print("\n2021 missing values:\n", df_2021.isna().sum())

> Note: VotingStationName was missing for 170 rows (2 voting districts) in the 2021 data. All other fields (VotingDistrict, RegisteredVoters, BallotType, vote counts) were complete for these rows. Since ward-level aggregation groups on VotingDistrict rather than station name, this has no effect on turnout calculations.

In [ ]:
# Ward ID format check - both years should use the same 'Ward XXXXXXXX' format
print(df_2016["Ward"].sample(5, random_state=1).tolist())
print(df_2021["Ward"].sample(5, random_state=1).tolist())

In [ ]:
# Unique ward counts per year
print("2016 unique wards (raw):", df_2016["Ward"].nunique())
print("2021 unique wards (raw):", df_2021["Ward"].nunique())

### Notes

